# 第2回　代表値の性質と、その嘘
## ―― 平均・中央値・最頻値、そして外れ値

統計学Ⅰ（B）　／　北星学園大学

今日も**▶を上から押すだけ**でよい（コードを書くのは第4回から）。注目するのは一つ ――

> 同じデータでも、**どの代表値を使うかで「物語」が変わる**。

In [ ]:
# 準備：ライブラリと、霊長類376種のデータを読み込む。▶ を押すだけ。
!pip install -q japanize-matplotlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import japanize_matplotlib  # noqa: F401
from scipy import stats

def _build_from_source():
    """公開データ（PanTHERIA）から、この授業で使う形に組み立て直す。"""
    SRC = "https://esapubs.org/archive/ecol/E090/184/PanTHERIA_1-0_WR05_Aug2008.txt"
    fam = {"Cercopithecidae":"オナガザル科","Cebidae":"オマキザル科","Pitheciidae":"サキ科",
           "Atelidae":"クモザル科","Cheirogaleidae":"コビトキツネザル科","Lemuridae":"キツネザル科",
           "Galagidae":"ガラゴ科","Hylobatidae":"テナガザル科","Indriidae":"インドリ科",
           "Lorisidae":"ロリス科","Lepilemuridae":"イタチキツネザル科","Aotidae":"ヨザル科",
           "Hominidae":"ヒト科","Tarsiidae":"メガネザル科","Daubentoniidae":"アイアイ科"}
    cols = {"MSW05_Binomial":"学名","MSW05_Genus":"属","5-1_AdultBodyMass_g":"体重g",
            "13-1_AdultHeadBodyLen_mm":"頭胴長mm","5-3_NeonateBodyMass_g":"新生児体重g",
            "10-2_SocialGrpSize":"集団サイズ","9-1_GestationLen_d":"妊娠期間日",
            "25-1_WeaningAge_d":"離乳日齢","3-1_AgeatFirstBirth_d":"初産日齢",
            "14-1_InterbirthInterval_d":"出産間隔日","15-1_LitterSize":"一腹産子数",
            "17-1_MaxLongevity_m":"最長寿命月","22-1_HomeRange_km2":"行動圏km2",
            "21-1_PopulationDensity_n/km2":"個体群密度","26-1_GR_Area_km2":"分布域km2",
            "6-2_TrophicLevel":"栄養段階","12-1_HabitatBreadth":"生息環境幅",
            "28-2_Temp_Mean_01degC":"平均気温01","28-1_Precip_Mean_mm":"月降水量mm"}
    src = pd.read_csv(SRC, sep="\t")
    p = src[src["MSW05_Order"] == "Primates"]
    out = p[list(cols)].rename(columns=cols)
    out.insert(1, "科", p["MSW05_Family"].map(fam))
    t = out.pop("平均気温01")
    out["平均気温C"] = np.where(t == -999, -999, (t / 10).round(1))
    return out.replace(-999, np.nan).sort_values("学名").reset_index(drop=True)

try:
    df = pd.read_csv("https://aonoa68.github.io/toukei-1/data/primates.csv")
except Exception:
    df = _build_from_source()

print("種数:", len(df), " 科数:", df["科"].nunique())
df.head()

---
## フック：「平均的な霊長類」は実在するか

**まず、コードを動かす前に頭の中で答えてほしい。**

> サルの仲間（霊長類）の**平均体重**は、何kgくらいだと思いますか。

ニホンザルが10kgくらい。小さいのもいるし、ゴリラもいる。……だいたい5kg？ 3kg？

**数字を出す前に、自分の答えを決めてください。** 決めたら次に進む。

In [ ]:
w = df["体重g"].dropna()

print(f"種数　　: {len(w)} 種")
print(f"平均値　: {w.mean():>9,.0f} g")
print(f"中央値　: {w.median():>9,.0f} g   ← 265種を重さ順に並べたとき真ん中の種")
print(f"最大値　: {w.max():>9,.0f} g   ← この種が平均を吊り上げている")
print(f"最小値　: {w.min():>9,.0f} g")

**平均 5,881g（約5.9kg）、中央値 3,006g（約3.0kg）。ほぼ2倍のひらきがある。**

あなたの直感はどちらに近かっただろうか。そして――**どちらが「ふつうのサル」なのか。**

犯人を見に行こう。

In [ ]:
print("重いほうから5種:")
print(df.nlargest(5, "体重g")[["学名", "科", "体重g"]].to_string(index=False))
print()
print("軽いほうから5種:")
print(df.nsmallest(5, "体重g")[["学名", "科", "体重g"]].to_string(index=False))

**ヒガシゴリラ 149kg、ネズミキツネザル 31g。その差は4,800倍。**

同じ「霊長類」というくくりの中に、これだけの幅がある。平均は、この幅を**たった1つの数字に押し込めた**結果である。グラフで見よう。

In [ ]:
plt.figure(figsize=(9,4))
plt.hist(w, bins=50, color="#80cbc4", edgecolor="white")
plt.axvline(w.mean(),   color="#e8503a", lw=2, label=f"平均 {w.mean():,.0f}g")
plt.axvline(w.median(), color="#1565c0", lw=2, ls="--", label=f"中央値 {w.median():,.0f}g")
plt.xlabel("体重（g）"); plt.ylabel("種数")
plt.title("右に長い裾を引く分布：平均が中央値より右へ引っ張られる")
plt.legend(); plt.show()

山は左側（数百g〜数kg）にあるのに、平均（赤線）は右にずれている。**分布が左右対称でないと、平均は山の位置を表さない。**

はるか右にいる大型類人猿――これが**外れ値**である。

> **そして重要なのは、この外れ値が「間違い」ではないこと。** ゴリラは本当に149kgある。
> データの誤りではなく、**世界がそういう形をしている**のだ。

---
## 外れ値は、平均をどれだけ動かすか

大型類人猿（ヒト科・7種）を仮に除いてみる。**平均と中央値がそれぞれどう動くか**に注目。

In [ ]:
no_ape = df[df["科"] != "ヒト科"]["体重g"].dropna()

print("            平均        中央値      種数")
print(f"全種　　  : {w.mean():>8,.0f} g   {w.median():>8,.0f} g   {len(w)}")
print(f"ヒト科除く: {no_ape.mean():>8,.0f} g   {no_ape.median():>8,.0f} g   {len(no_ape)}")
print()
print(f"平均は   {w.mean()-no_ape.mean():>6,.0f} g 動いた（外れ値に弱い）")
print(f"中央値は {w.median()-no_ape.median():>6,.0f} g しか動かない（外れ値に強い＝頑健）")
print()
print(f"→ 動いた量の比は約 {(w.mean()-no_ape.mean())/(w.median()-no_ape.median()):.1f} 倍")

**たった7種（全体の2.6%）を抜いただけで平均は1,754g動く**のに、**中央値は209gしか動かない**。
中央値は外れ値に強い（これを「頑健 robust」と呼ぶ）。

> ⚠️ **では外れ値は消していいのか？**

> 答えは**「場合による」**。
> - ゴリラの149kg は**本物**。「平均を下げたいから」で消すのは**捏造**である
> - 一方、体重が **−999g** と書かれていたら、それは欠測を表す符号で、**明らかな誤り**（このデータの元ファイルには実際に入っている）
> - **見分けるのはコードではなく、対象を知っている人間**である。そして外れ値の扱いは、**理由を説明できなければならない**

---
## 最頻値 ―― いちばん「よくいる」のはどこか

平均でも中央値でもなく、**いちばん種数が多い帯**を見る。
体重のように何桁にもまたがる数字は、**対数の目盛りで区切る**と山が見える。

In [ ]:
import numpy as np
h, edges = np.histogram(np.log10(w), bins=20)

print("対数目盛りでの分布")
for i in range(20):
    print(f"  {10**edges[i]:>8,.0f}〜{10**edges[i+1]:>8,.0f}g {h[i]:>3d} " + "█"*h[i])

i = h.argmax()
print()
print(f"最も種数が多い帯: {10**edges[i]:,.0f}g 〜 {10**edges[i+1]:,.0f}g　（{h[i]}種）")

**山が2つある。**

- 400g〜1,400g あたりに、なだらかな山（小型のサル・キツネザルの仲間）
- **5,039〜7,696g に、飛び抜けた山（55種）**

そして、ここからが今日いちばん大事なところである。**3つの代表値がどこに落ちるかを見る。**

In [ ]:
i = h.argmax()
j = np.searchsorted(edges, np.log10(w.median())) - 1

print(f"平均値　: {w.mean():>9,.0f} g  → 最も種数の多い帯（{10**edges[i]:,.0f}〜{10**edges[i+1]:,.0f}g・{h[i]}種）の中")
print(f"中央値　: {w.median():>9,.0f} g  → {10**edges[j]:,.0f}〜{10**edges[j+1]:,.0f}g の帯（{h[j]}種）")
print()
print("→ 中央値は、2つの山の「谷間」に落ちている")

**中央値 3,006g のあたりに、そんなに種はいない。** 2つの山にはさまれた谷である。

つまり ――

> **平均も中央値も、「よくいる霊長類の姿」を指していない。**

平均5.9kgは大型のサルが多い山に引っ張られた結果であり、中央値3.0kgは「上から数えても下から数えても真ん中」というだけで、**そこに実際に多くの種がいるわけではない。**

なぜ山が2つできるのか。科ごとに見ると分かる。

In [ ]:
fam = (df.groupby("科")["体重g"]
         .agg(種数="count", 中央値="median")
         .dropna()
         .sort_values("中央値"))
fam.round(0)

**オナガザル科だけで92種**（体重の記録がある265種の35%）。この科の中央値が7.2kgで、右側の山の正体はこれである。左側の山は、コビトキツネザル科・ガラゴ科・オマキザル科といった小型のグループ。

> **1つの数字で全体を代表させようとしたことが、そもそも無理だった。**
> 分布が二山なら、答えは「平均◯kg」ではなく「**大きく2つのグループがある**」である。

> ⚠️ **もうひとつ。この265種は「体重が測られた種」だけである。**
> よく研究された科（オナガザル科）ほど多く含まれ、記録の少ない科は薄い。**データの形は、世界の形であると同時に、研究の歴史の形でもある。**

---
## 対照実験：左右対称なデータなら、3つは一致する

体重は歪んでいたから3つがズレた。では**ほぼ左右対称**な `妊娠期間日` ではどうか。

In [ ]:
g = df["妊娠期間日"].dropna()

print(f"妊娠期間  平均 {g.mean():.1f}日 / 中央値 {g.median():.1f}日 / 種数 {len(g)}")
print("→ 平均と中央値がほぼ一致。対称な分布では『どれを使っても同じ』")
print()

plt.figure(figsize=(8,3.5))
plt.hist(g, bins=25, color="#80cbc4", edgecolor="white")
plt.axvline(g.mean(),   color="#e8503a", lw=2, label=f"平均 {g.mean():.0f}日")
plt.axvline(g.median(), color="#1565c0", lw=2, ls="--", label=f"中央値 {g.median():.0f}日")
plt.title("ほぼ対称な分布：平均と中央値がほぼ重なる")
plt.xlabel("妊娠期間（日）"); plt.ylabel("種数"); plt.legend(); plt.show()

同じデータセットの中でも、**列によって分布の形はまったく違う**。
「このデータは平均を使ってよいか」は、列ごとに判断することになる。

---
## もうひとつの手：目盛りを変える

体重を**対数の目盛り**で描き直すと、どうなるか。

In [ ]:
from scipy import stats

plt.figure(figsize=(9,4))
plt.hist(np.log10(w), bins=40, color="#80cbc4", edgecolor="white")
plt.axvline(np.log10(w.median()), color="#1565c0", lw=2, ls="--",
            label=f"中央値 {w.median():,.0f}g")
plt.xticks([1,2,3,4,5], ["10g","100g","1kg","10kg","100kg"])
plt.xlabel("体重（対数目盛り）"); plt.ylabel("種数")
plt.title("目盛りを変えると、左右対称に近い山が現れる")
plt.legend(); plt.show()

print(f"そのまま　　 の歪み（歪度）: {stats.skew(w):>6.2f}   ← 0から遠いほど左右非対称")
print(f"対数にした後 の歪み　　　　: {stats.skew(np.log(w)):>6.2f}")

**データは1ミリも変えていない。目盛りを変えただけ**で、歪度が 7.60 から −0.40 になった。

「このデータは歪んでいる」というのは、**データだけの性質ではなく、どの尺度で見るかとの組み合わせ**である。
体重・年収・都市人口・企業規模のように何桁にもまたがる量は、対数で見るのが自然なことが多い。

---
## 今日のまとめ

| 代表値 | 強み | 弱み | 向いている場面 |
|---|---|---|---|
| 平均値 | 全データを使う・計算しやすい | **外れ値・歪みに弱い** | 左右対称なデータ |
| 中央値 | 外れ値に強い（頑健） | 値の大小の情報を一部捨てる | **体重・年収など歪んだデータ** |
| 最頻値 | カテゴリにも使える | 連続データでは区切り方に左右される | アンケートの選択肢・人気 |

> **代表値の選択は、中立ではない。** どれを選ぶかは、すでに一つの**主張**である。
> だから――**数字を見たら、まず分布の形を疑え**。

そして今日の本当の教訓は、その先にある。

> **この分布では、平均も中央値も最頻値も「ふつうの霊長類」を指していなかった。**
> 山が2つあるとき、1つの代表値で答えようとすること自体が間違いである。

あなたの直感は当たっただろうか。**「平均5,881gの霊長類」は実在しない。**そして**「中央値3,006gの霊長類」も、あまりいない。**

代表値を出す前にヒストグラムを描く。順番を逆にすると、今日のような見落としが起きる。

**課題（Moodle）**：シナリオごとに「報告すべき代表値とその理由」を答える。

---

!!! quote "このデータの出典"
    Jones, K.E. et al. (2009) PanTHERIA: a species-level database of life history,
    ecology, and geography of extant and recently extinct mammals.
    *Ecology* 90(9): 2648. Ecological Archives E090-184.

    霊長類376種の行だけを抜き出し、列を選び、気温の単位を直したもの。値は変えていない。